# Questions 6.1 to 6.4

### Imports

In [7]:
import config as cfg
import joblib
import src.loading as ld
import src.normalise as norm
import src.features as ftrs
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

### Loading DataSet (Dev + Test Sets)
All 24 actors are loaded (`include_tests=True`).

In [8]:
full_df = ld.load_frames(include_tests=True)

### Normalising Data

In [9]:
full_df_n = norm.normalise_d(full_df)

### Feature Table & Extracting Values

In [10]:
full_table = ftrs.build_feature_table(full_df_n)
y_full = full_table.index.get_level_values("emotion").to_numpy()

print("Feature Table Shape:", full_table.shape)
print("Number of unique actors:", full_table.index.get_level_values("actor").nunique())

Feature Table Shape: (1440, 55)
Number of unique actors: 24


## 6.1 Final Model

### Fitting the Pipeline of 5.6 on the 1440 Videos
The pipeline delivered in 5.6, `StandardScaler → SVC(C = 1)`, with `probability=True` so that it returns the probability of each emotion.

In [11]:
final_model = Pipeline([("scaler", StandardScaler()),
                        ("svc", SVC(kernel="rbf", C=cfg.C_FIXED, probability=True, random_state=cfg.SEED))])
final_model.fit(full_table, y_full);

/Users/andreleao/Documents/MIA/1ANO/IAC/Emotion_Recognition/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


### Saving the Model

In [12]:
joblib.dump(final_model, cfg.ROOT / "app" / "model.joblib")

['/Users/andreleao/Documents/MIA/1ANO/IAC/Emotion_Recognition/app/model.joblib']

### Answer 6.1

The pipeline chosen in 5.6 is fitted on all 1440 videos of the 24 actors and saved to `app/model.joblib`.

In `classifier.py`, `load_model` reads `model.joblib`, and `predict` receives the frames of the clip in pixels, normalises them with `normalise_d`, computes the 55 features with `video_features`, and returns the probability of each emotion from `predict_proba`. Both functions are imported from our `src` modules, the same used for training, placed inside the `app` folder.

An SVC returns probabilities only with `probability=True`: it then fits a sigmoid on its decision values with an internal cross-validation (Platt scaling).